In [3]:
import pandas as pd
from pathlib import Path
import re
# --------------------------
# 0) Chemins du projet
# --------------------------
CODE_DIR = Path().resolve()              # .../SYNCOGEST/Code/Codes VICON
PROJECT_ROOT = CODE_DIR.parent.parent    # .../SYNCOGEST
DATA_DIR = PROJECT_ROOT / "DATA"

VICON_DIR = DATA_DIR / "VICON_CSV"
EXCEL_DIR = DATA_DIR / "Excels_code"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("VICON_DIR =", VICON_DIR)
print("EXCEL_DIR =", EXCEL_DIR)

# --- chemins ---
qdm_path = EXCEL_DIR / "vicon_QDM_wrists_head_mm.xlsx"
sw_path  = EXCEL_DIR / "vicon_shoulder_width_by_csv.xlsx"

# --- charge ---
qdm = pd.read_excel(qdm_path)
sw  = pd.read_excel(sw_path)

# =========================
# 1) Créer une clé vidéo propre
# =========================

# QDM: colonne 'file' = "SEATEDD01.csv"
qdm["video_id"] = qdm["file"].astype(str).str.replace(".csv", "", regex=False)

# Shoulder width: colonne 'csv' = chemin complet
sw["video_id"] = sw["csv"].astype(str).apply(lambda x: Path(x).stem)

# =========================
# 2) Merge (ordre indépendant)
# =========================
out = qdm.merge(
    sw[[
        "video_id",
        "P1_shoulder_width_median_mm",
        "P2_shoulder_width_median_mm"
    ]],
    on="video_id",
    how="left"
)

# contrôle
missing = out["P1_shoulder_width_median_mm"].isna().sum()
print(f"Lignes sans largeur épaules trouvée : {missing}/{len(out)}")
if missing > 0:
    print(out.loc[out["P1_shoulder_width_median_mm"].isna(), ["video_id"]])

# =========================
# 3) Normalisation QDM / largeur épaules
# =========================

# --- poignets ---
out["P1_QDM_WRISTS_norm"] = (
    out["P1_QDM_WRISTS_mm"] / out["P1_shoulder_width_median_mm"]
)

out["P2_QDM_WRISTS_norm"] = (
    out["P2_QDM_WRISTS_mm"] / out["P2_shoulder_width_median_mm"]
)

# --- tête ---
out["P1_QDM_HEAD_norm"] = (
    out["P1_QDM_HEAD_mm"] / out["P1_shoulder_width_median_mm"]
)

out["P2_QDM_HEAD_norm"] = (
    out["P2_QDM_HEAD_mm"] / out["P2_shoulder_width_median_mm"]
)

# =========================
# 4) Sauvegarde
# =========================
out_path = EXCEL_DIR / "vicon_QDM_wrists_head_normByShoulders.xlsx"
out.to_excel(out_path, index=False)
print("✅ Saved:", out_path)

PROJECT_ROOT = /Users/matysprecloux/Desktop/SYNCOGEST
VICON_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VICON_CSV
EXCEL_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code
Lignes sans largeur épaules trouvée : 0/60
✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/vicon_QDM_wrists_head_normByShoulders.xlsx
